In [6]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import bokeh.plotting as bk
from bokeh.models import HoverTool
import unittest
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# File paths
dataset_train = "C:\\Users\\Nutzer\\OneDrive\\Documents\\Marketing updates\\Python\\Written assignment\\Datasets1\\train.csv"
dataset_test = "C:\\Users\\Nutzer\\OneDrive\\Documents\\Marketing updates\\Python\\Written assignment\\Datasets1\\test.csv"
dataset_ideal = "C:\\Users\\Nutzer\\OneDrive\\Documents\\Marketing updates\\Python\\Written assignment\\Datasets1\\ideal.csv"

class DataHandler:
    """Handles data loading and database creation."""
    def __init__(self, db_name="data_analysis.db"):
        self.engine = create_engine(f'sqlite:///{db_name}')
    
    def load_data_to_db(self, data, table_name):
        """Load data into a SQLite database table."""
        try:
            data.to_sql(table_name, self.engine, if_exists='replace', index=False)
            logging.info(f"Data loaded into table '{table_name}' successfully.")
            return True
        except Exception as e:
            logging.error(f"Error loading data to database: {e}")
            return False
    
    def load_data_from_csv(self, file_path):
        """Load data from a CSV file."""
        try:
            data = pd.read_csv(file_path)
            logging.info(f"Data loaded from {file_path} successfully.")
            logging.info(f"Columns in the dataset: {data.columns.tolist()}")  # Log column names
            return data
        except Exception as e:
            logging.error(f"Error loading file {file_path}: {e}")
            return None

class DataValidator(DataHandler):
    """Extends DataHandler to include data validation before loading into the database."""
    def validate_data(self, data):
        """Validate that the data contains no null values and has the required columns."""
        if data.isnull().values.any():
            raise ValueError("Data contains null values.")
        
        # Convert column names to lowercase for case-insensitive comparison
        lowercase_columns = [col.lower() for col in data.columns]
        
        # Check for 'x' and at least one 'y' column
        if 'x' not in lowercase_columns:
            raise ValueError("Data must contain an 'x' column.")
        
        y_columns = [col for col in lowercase_columns if col.startswith('y')]
        if not y_columns:
            raise ValueError("Data must contain at least one 'y' column.")
        
        logging.info(f"Validated columns: 'x' and {y_columns}")
        return True

class IdealFunctionSelector(DataHandler):
    """Selects the best fitting ideal functions based on least squares criterion."""
    def find_best_fit(self, training_data, ideal_functions):
        """Find the best fitting ideal functions for the training data."""
        selected_functions = []
        for i in range(1, 5):  # Assuming there are 4 Y columns (y1, y2, y3, y4)
            min_error = float('inf')
            best_function = None
            for col in ideal_functions.columns[1:]:  # Skip the 'x' column
                error = np.sum((training_data[f'y{i}'] - ideal_functions[col]) ** 2)
                if error < min_error:
                    min_error = error
                    best_function = col
            selected_functions.append(best_function)
        return selected_functions

class TestMapper(DataHandler):
    """Maps test data to the closest ideal function within sqrt(2) deviation criteria."""
    def map_test_data(self, test_data, ideal_functions, selected_functions):
        """Map test data to the closest ideal function."""
        results = []
        for _, row in test_data.iterrows():
            x, y = row['x'], row['y']
            best_fit, min_deviation = None, float('inf')
            for func in selected_functions:
                y_ideal = ideal_functions.loc[ideal_functions['x'] == x, func].values[0]
                deviation = abs(y - y_ideal)
                if deviation < min_deviation * np.sqrt(2):
                    min_deviation = deviation
                    best_fit = func
            results.append((x, y, min_deviation, best_fit))
        return results

def visualize_data(training_data, test_data, ideal_functions, selected_functions):
    """Visualize the training data, test data, and selected ideal functions."""
    bk.output_notebook()
    p = bk.figure(title='Data Visualization', x_axis_label='X', y_axis_label='Y')
    colors = ['red', 'green', 'blue', 'purple']
    
    # Plot training data
    for i in range(1, 5):  # Assuming there are 4 Y columns (y1, y2, y3, y4)
        p.scatter(training_data['x'], training_data[f'y{i}'], legend_label=f'Training y{i}', color=colors[i-1])
    
    # Plot selected ideal functions
    for idx, func in enumerate(selected_functions):
        p.line(ideal_functions['x'], ideal_functions[func], legend_label=f'Ideal {func}', color=colors[idx], line_width=2)
    
    # Plot test data
    p.scatter(test_data['x'], test_data['y'], legend_label='Test Data', color='black', size=7)
    
    p.add_tools(HoverTool())
    bk.show(p)

class TestFunctions(unittest.TestCase):
    """Unit tests for the functions."""
    def test_least_square_selection(self):
        """Test the least square selection method."""
        dummy_training = pd.DataFrame({'x': [1, 2, 3], 'y1': [2, 4, 6]})
        dummy_ideal = pd.DataFrame({'x': [1, 2, 3], 'y1': [2.1, 3.9, 6.1], 'y2': [1, 2, 3]})
        selector = IdealFunctionSelector()
        result = selector.find_best_fit(dummy_training, dummy_ideal)
        self.assertEqual(result[0], 'y1')
    
    def test_mapping(self):
        """Test the mapping of test data to ideal functions."""
        dummy_test = pd.DataFrame({'x': [1, 2, 3], 'y': [2, 4, 6]})
        dummy_ideal = pd.DataFrame({'x': [1, 2, 3], 'y1': [2, 4, 6], 'y2': [1, 2, 3]})
        mapper = TestMapper()
        result = mapper.map_test_data(dummy_test, dummy_ideal, ['y1', 'y2'])
        self.assertEqual(result[0][3], 'y1')

if __name__ == "__main__":
    try:
        # Initialize DataHandler
        data_handler = DataHandler()
        
        # Load and validate data
        training_data = data_handler.load_data_from_csv(dataset_train)
        ideal_data = data_handler.load_data_from_csv(dataset_ideal)
        test_data = data_handler.load_data_from_csv(dataset_test)
        
        if training_data is not None and ideal_data is not None and test_data is not None:
            # Validate training data
            validator = DataValidator()
            validator.validate_data(training_data)
            
            # Load data into SQLite database
            data_handler.load_data_to_db(training_data, 'training_data')
            data_handler.load_data_to_db(ideal_data, 'ideal_functions')
            data_handler.load_data_to_db(test_data, 'test_data')
            
            # Select best fitting ideal functions
            selector = IdealFunctionSelector()
            selected_functions = selector.find_best_fit(training_data, ideal_data)
            
            # Map test data to ideal functions
            mapper = TestMapper()
            mapped_results = mapper.map_test_data(test_data, ideal_data, selected_functions)
            
            # Save mapped results to SQLite database
            mapped_results_df = pd.DataFrame(mapped_results, columns=['x', 'y', 'delta y', 'no. of ideal func'])
            data_handler.load_data_to_db(mapped_results_df, 'mapped_results')
            
            # Visualize data
            visualize_data(training_data, test_data, ideal_data, selected_functions)
        else:
            logging.error("Failed to load one or more datasets.")
    except Exception as e:
        logging.error(f"An error occurred: {e}")

2025-03-21 23:57:38,294 - INFO - Data loaded from C:\Users\Nutzer\OneDrive\Documents\Marketing updates\Python\Written assignment\Datasets1\train.csv successfully.
2025-03-21 23:57:38,299 - INFO - Columns in the dataset: ['x', 'y1', 'y2', 'y3', 'y4']
2025-03-21 23:57:38,321 - INFO - Data loaded from C:\Users\Nutzer\OneDrive\Documents\Marketing updates\Python\Written assignment\Datasets1\ideal.csv successfully.
2025-03-21 23:57:38,322 - INFO - Columns in the dataset: ['x', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12', 'y13', 'y14', 'y15', 'y16', 'y17', 'y18', 'y19', 'y20', 'y21', 'y22', 'y23', 'y24', 'y25', 'y26', 'y27', 'y28', 'y29', 'y30', 'y31', 'y32', 'y33', 'y34', 'y35', 'y36', 'y37', 'y38', 'y39', 'y40', 'y41', 'y42', 'y43', 'y44', 'y45', 'y46', 'y47', 'y48', 'y49', 'y50']
2025-03-21 23:57:38,327 - INFO - Data loaded from C:\Users\Nutzer\OneDrive\Documents\Marketing updates\Python\Written assignment\Datasets1\test.csv successfully.
2025-03-21 23:57:38,3

Loading BokehJS ...

In [1]:
# Cell 1: Imports
from data_handler import DataHandler
import pandas as pd

# Cell 2: Initialize and test
handler = DataHandler()  # Uses data_analysis.db
test_data = pd.DataFrame({'x': [1,2,3], 'y': [4,5,6]})
test_data.to_sql('test_table', handler.engine, if_exists='replace')

# Cell 3: Calculate stats
stats = handler.calculate_statistics("test_table")
stats

,index,x,y
count,3.0,3.0,3.0
mean,1.0,2.0,5.0
std,1.0,1.0,1.0
min,0.0,1.0,4.0
25%,0.5,1.5,4.5
50%,1.0,2.0,5.0
75%,1.5,2.5,5.5
max,2.0,3.0,6.0
